# DVF Diffusion — Clean Rebuild

## Section 0 — Imports and paths

In [11]:
import os
import math
import random
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm.auto import tqdm

# ── Paths ─────────────────────────────────────────────────────────────────────
SYNTH_TRAIN_FIELD_DIR = '/mimer/NOBACKUP/groups/caim1/dafne/datasets/synthetic/train/b'
SYNTH_TEST_FIELD_DIR  = '/mimer/NOBACKUP/groups/caim1/dafne/datasets/synthetic/test/b'
SAVE_DIR              = '/mimer/NOBACKUP/groups/caim1/dafne/checkpoints'
VIS_DIR               = '/mimer/NOBACKUP/groups/caim1/dafne/visual'
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(VIS_DIR,  exist_ok=True)

# ── Global constants (derived from data inspection, do not change) ─────────────
# Per-channel normalization (derived from full training set statistics)
# Channel 0 (dx LR): worst abs = 30.7 → 35.0
# Channel 1 (dy AP): worst abs = 70.9 → 75.0  
# Channel 2 (dz SI): worst abs = 73.2 → 80.0
DVF_NORM = np.array([35.0, 75.0, 80.0], dtype=np.float32)  # (3,)
DVF_SHAPE  = (3, 256, 256, 128)   # (C, H, W, D)
H, W, D    = 256, 256, 128

# Slice position margins: don't condition on slices within 20% of the edge
# so the model always has some context on both sides of the conditioning plane
H_MARGIN   = int(0.20 * H)  # = 51
W_MARGIN   = int(0.20 * W)  # = 51

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'H_MARGIN={H_MARGIN}, W_MARGIN={W_MARGIN}')
print(f'Coronal slice range:  h_idx in [{H_MARGIN}, {H - H_MARGIN - 1}]')
print(f'Sagittal slice range: w_idx in [{W_MARGIN}, {W - W_MARGIN - 1}]')

Device: cuda
H_MARGIN=51, W_MARGIN=51
Coronal slice range:  h_idx in [51, 204]
Sagittal slice range: w_idx in [51, 204]


## Section 1 — Data inspection

In [12]:
# ── 1a: verify file counts ─────────────────────────────────────────────────────
train_files = sorted(f for f in os.listdir(SYNTH_TRAIN_FIELD_DIR) if f.endswith('.npy'))
test_files  = sorted(f for f in os.listdir(SYNTH_TEST_FIELD_DIR)  if f.endswith('.npy'))
print(f'Train files: {len(train_files)}')
print(f'Test  files: {len(test_files)}')
print(f'Train first/last: {train_files[0]} / {train_files[-1]}')
print(f'Test  first/last: {test_files[0]}  / {test_files[-1]}')

Train files: 500
Test  files: 100
Train first/last: field_00000.npy / field_00499.npy
Test  first/last: field_00000.npy  / field_00099.npy


In [15]:
# ── 1b: verify normalization constant covers the data ──────────────────────────
rng = np.random.default_rng(42)
sample_idx = rng.choice(len(train_files), size=30, replace=False)

exceed_count = 0
total_voxels = 0
abs_maxes    = []

for i in sample_idx:
    dvf = np.load(os.path.join(SYNTH_TRAIN_FIELD_DIR, train_files[i]))
    # DVF_NORM is (3,) → reshape to (3,1,1,1) for broadcast against (3,H,W,D)
    exceed_count += (np.abs(dvf) > DVF_NORM[:, None, None, None]).sum()
    total_voxels += dvf.size
    abs_maxes.append(np.abs(dvf).max())

print(f'DVF_NORM = {DVF_NORM}')
print(f'Voxels exceeding DVF_NORM: {exceed_count}/{total_voxels} '
      f'({100*exceed_count/total_voxels:.3f}%)')
print(f'Max absolute value seen:   {max(abs_maxes):.4f}')

# per-channel check
dvf0 = np.load(os.path.join(SYNTH_TRAIN_FIELD_DIR, train_files[0]))
dvf0_norm = dvf0 / DVF_NORM[:, None, None, None]
print('\nPer-channel range after normalization (sample 0):')
for c in range(3):
    print(f'  channel {c}: [{dvf0_norm[c].min():.4f}, {dvf0_norm[c].max():.4f}]  '
          f'(expect within [-1, 1])')

DVF_NORM = [35. 75. 80.]
Voxels exceeding DVF_NORM: 0/754974720 (0.000%)
Max absolute value seen:   68.8241

Per-channel range after normalization (sample 0):
  channel 0: [-0.5161, 0.4320]  (expect within [-1, 1])
  channel 1: [-0.0118, 0.7028]  (expect within [-1, 1])
  channel 2: [-0.5073, 0.1148]  (expect within [-1, 1])


In [16]:
# ── 1c: visualise one raw sample (before normalisation) ───────────────────────
dvf0 = np.load(os.path.join(SYNTH_TRAIN_FIELD_DIR, train_files[0]))
# shape: (3, H, W, D) = (3, 256, 256, 128)
# show the middle axial slice (D//2) for each component
mid_d = D // 2  # = 64

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
comp_names = ['dx (channel 0)', 'dy (channel 1)', 'dz (channel 2)']
for c in range(3):
    sl   = dvf0[c, :, :, mid_d]   # (H, W) axial slice
    vmax = np.percentile(np.abs(sl), 99)
    im   = axes[c].imshow(sl, cmap='RdBu_r', vmin=-vmax, vmax=vmax, origin='lower')
    axes[c].set_title(f'{comp_names[c]}\nmin={sl.min():.1f} max={sl.max():.1f}')
    axes[c].axis('off')
    plt.colorbar(im, ax=axes[c], fraction=0.046, pad=0.04)
plt.suptitle(f'Raw DVF — sample 0, axial slice at D={mid_d}', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(VIS_DIR, 'inspect_raw_dvf.png'), dpi=120, bbox_inches='tight')
plt.show()
print('Saved to', os.path.join(VIS_DIR, 'inspect_raw_dvf.png'))

Saved to /mimer/NOBACKUP/groups/caim1/dafne/visual/inspect_raw_dvf.png


In [17]:
# ── 1d: show what a random coronal and sagittal slice look like ────────────────
# This is exactly what the model will receive as conditioning input
rng2   = np.random.default_rng(7)
h_idx  = rng2.integers(H_MARGIN, H - H_MARGIN)   # random coronal position
w_idx  = rng2.integers(W_MARGIN, W - W_MARGIN)   # random sagittal position

cor_slice = dvf0[:, h_idx, :, :]   # (3, W, D) = (3, 256, 128)
sag_slice = dvf0[:, :, w_idx, :]   # (3, H, D) = (3, 256, 128)

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for c in range(3):
    # coronal row
    sl   = cor_slice[c]   # (W, D)
    vmax = np.percentile(np.abs(sl), 99)
    im   = axes[0, c].imshow(sl.T, cmap='RdBu_r', vmin=-vmax, vmax=vmax, origin='lower')
    axes[0, c].set_title(f'Coronal h={h_idx} — {comp_names[c]}')
    axes[0, c].set_xlabel('W axis'); axes[0, c].set_ylabel('D axis')
    plt.colorbar(im, ax=axes[0, c], fraction=0.046, pad=0.04)

    # sagittal row
    sl   = sag_slice[c]   # (H, D)
    vmax = np.percentile(np.abs(sl), 99)
    im   = axes[1, c].imshow(sl.T, cmap='RdBu_r', vmin=-vmax, vmax=vmax, origin='lower')
    axes[1, c].set_title(f'Sagittal w={w_idx} — {comp_names[c]}')
    axes[1, c].set_xlabel('H axis'); axes[1, c].set_ylabel('D axis')
    plt.colorbar(im, ax=axes[1, c], fraction=0.046, pad=0.04)

plt.suptitle(f'Conditioning slices — sample 0 (raw, not normalised)', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(VIS_DIR, 'inspect_slices.png'), dpi=120, bbox_inches='tight')
plt.show()
print(f'Coronal slice shape:  {cor_slice.shape}  (expect (3, 256, 128))')
print(f'Sagittal slice shape: {sag_slice.shape}  (expect (3, 256, 128))')

Coronal slice shape:  (3, 256, 128)  (expect (3, 256, 128))
Sagittal slice shape: (3, 256, 128)  (expect (3, 256, 128))


In [18]:
# Check per-channel mean across 50 samples
for c in range(3):
    means = []
    for f in sorted(os.listdir(SYNTH_TRAIN_FIELD_DIR))[:50]:
        dvf = np.load(os.path.join(SYNTH_TRAIN_FIELD_DIR, f))
        means.append(dvf[c].mean())
    print(f'Channel {c}: mean={np.mean(means):.3f}  std={np.std(means):.3f}')

Channel 0: mean=-0.257  std=0.176
Channel 1: mean=8.300  std=1.218
Channel 2: mean=-13.003  std=3.088


In [19]:
import numpy as np
import os

field_dir = "/mimer/NOBACKUP/groups/caim1/dafne/datasets/synthetic/train/b"
files = sorted(f for f in os.listdir(field_dir) if f.endswith('.npy'))

# accumulate per-channel min/max/mean across ALL training files
ch_mins  = [[], [], []]
ch_maxs  = [[], [], []]
ch_means = [[], [], []]
ch_stds  = [[], [], []]

for f in files:
    dvf = np.load(os.path.join(field_dir, f))  # (3, H, W, D)
    for c in range(3):
        ch_mins[c].append(dvf[c].min())
        ch_maxs[c].append(dvf[c].max())
        ch_means[c].append(dvf[c].mean())
        ch_stds[c].append(dvf[c].std())

print("Per-channel statistics across ALL 500 training samples:")
for c in range(3):
    print(f"\nChannel {c}:")
    print(f"  min:  {min(ch_mins[c]):.4f}  (worst case minimum)")
    print(f"  max:  {max(ch_maxs[c]):.4f}  (worst case maximum)")
    print(f"  mean of means: {np.mean(ch_means[c]):.4f}")
    print(f"  mean of stds:  {np.mean(ch_stds[c]):.4f}")
    print(f"  99th pct of abs values: {np.percentile(np.abs(ch_mins[c] + ch_maxs[c]), 99):.4f}")

Per-channel statistics across ALL 500 training samples:

Channel 0:
  min:  -30.6616  (worst case minimum)
  max:  26.7028  (worst case maximum)
  mean of means: -0.3004
  mean of stds:  3.5583
  99th pct of abs values: 24.9560

Channel 1:
  min:  -0.8863  (worst case minimum)
  max:  70.8941  (worst case maximum)
  mean of means: 8.2196
  mean of stds:  8.0379
  99th pct of abs values: 62.4586

Channel 2:
  min:  -73.1809  (worst case minimum)
  max:  16.9258  (worst case maximum)
  mean of means: -13.1359
  mean of stds:  10.9478
  99th pct of abs values: 69.8208


## Section 1 checkpoint
- Train=500, Test=100 ✓
- Less than 1% of voxels exceed DVF_NORM=75 ✓
- Coronal slice shape is (3, 256, 128) ✓
- Sagittal slice shape is (3, 256, 128) ✓
- The raw DVF images look like smooth displacement fields (not noise) ✓

## Section 2 — Dataset

In [20]:
class SyntheticDVFDataset(Dataset):
    """
    Loads DVF volumes and extracts random coronal + sagittal slices on the fly.
    The stored slice files are NOT used — slices are always extracted from the
    full DVF at random positions so the model learns position-awareness.

    Returns (normalised after loading):
        dvf:       (3, H, W, D) float32, values in approx [-1, 1]
        cor:       (3, W, D)    float32, coronal slice  at h_idx
        sag:       (3, H, D)    float32, sagittal slice at w_idx
        h_norm:    scalar float32, h_idx / (H-1)  in [0, 1]
        w_norm:    scalar float32, w_idx / (W-1)  in [0, 1]
    """
    def __init__(self, field_dir, h_margin=H_MARGIN, w_margin=W_MARGIN,
                 norm=DVF_NORM, seed=None):
        self.field_dir = field_dir
        self.files     = sorted(f for f in os.listdir(field_dir) if f.endswith('.npy'))
        self.h_margin  = h_margin
        self.w_margin  = w_margin
        self.norm      = norm
        self.rng       = np.random.default_rng(seed)  # None = random seed

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        dvf = np.load(os.path.join(self.field_dir, self.files[idx])).astype(np.float32)
        # dvf shape: (3, H, W, D)

        # norm is (3,) — broadcast over (3, H, W, D)
        dvf = dvf / self.norm[:, None, None, None]

        C, H, W, D = dvf.shape

        # random slice positions within the allowed margin
        h_idx = self.rng.integers(self.h_margin, H - self.h_margin)
        w_idx = self.rng.integers(self.w_margin, W - self.w_margin)

        cor = dvf[:, h_idx, :, :]   # (3, W, D)
        sag = dvf[:, :, w_idx, :]   # (3, H, D)

        h_norm = np.float32(h_idx / (H - 1))
        w_norm = np.float32(w_idx / (W - 1))

        return (
            torch.from_numpy(dvf),
            torch.from_numpy(cor),
            torch.from_numpy(sag),
            torch.tensor(h_norm),
            torch.tensor(w_norm),
        )

In [21]:
# ── 2a: verify dataset output ──────────────────────────────────────────────────
ds = SyntheticDVFDataset(SYNTH_TRAIN_FIELD_DIR, seed=0)
dvf, cor, sag, h_norm, w_norm = ds[0]

print(f'dvf shape:   {tuple(dvf.shape)}   (expect (3, 256, 256, 128))')
print(f'cor shape:   {tuple(cor.shape)}   (expect (3, 256, 128))')
print(f'sag shape:   {tuple(sag.shape)}   (expect (3, 256, 128))')
print(f'h_norm:      {h_norm.item():.4f}  (expect in [{H_MARGIN/(H-1):.3f}, {(H-H_MARGIN-1)/(H-1):.3f}])')
print(f'w_norm:      {w_norm.item():.4f}  (expect in [{W_MARGIN/(W-1):.3f}, {(W-W_MARGIN-1)/(W-1):.3f}])')
print(f'dvf range:   [{dvf.min():.4f}, {dvf.max():.4f}]  (expect approx [-1, 1])')

# verify that cor is actually a slice of dvf at h_idx
h_idx_recovered = round(h_norm.item() * (H - 1))
w_idx_recovered = round(w_norm.item() * (W - 1))
assert torch.allclose(cor, dvf[:, h_idx_recovered, :, :]), 'Coronal slice mismatch!'
assert torch.allclose(sag, dvf[:, :, w_idx_recovered, :]), 'Sagittal slice mismatch!'
print('\nSlice consistency check: PASSED')

dvf shape:   (3, 256, 256, 128)   (expect (3, 256, 256, 128))
cor shape:   (3, 256, 128)   (expect (3, 256, 128))
sag shape:   (3, 256, 128)   (expect (3, 256, 128))
h_norm:      0.7098  (expect in [0.200, 0.800])
w_norm:      0.5843  (expect in [0.200, 0.800])
dvf range:   [-0.5161, 0.7028]  (expect approx [-1, 1])

Slice consistency check: PASSED


In [36]:
# ── 2b: check that different calls give different slice positions ───────────────
# (important: confirms randomisation is working)
print('h_norm and w_norm for samples 0-4 (should vary):')
for i in range(5):
    _, _, _, h, w = ds[i]
    print(f'  sample {i}: h_norm={h.item():.4f}  w_norm={w.item():.4f}')

# also check that calling __getitem__ twice on the SAME index gives different slices
# (because rng advances)
_, _, _, h0a, w0a = ds[0]
_, _, _, h0b, w0b = ds[0]
print(f'\nSame index called twice: h_norm={h0a.item():.4f} vs {h0b.item():.4f}')

h_norm and w_norm for samples 0-4 (should vary):
  sample 0: h_norm=0.7098  w_norm=0.3059
  sample 1: h_norm=0.2510  w_norm=0.7176
  sample 2: h_norm=0.2118  w_norm=0.5255
  sample 3: h_norm=0.2471  w_norm=0.3804
  sample 4: h_norm=0.4902  w_norm=0.4549

Same index called twice: h_norm=0.4431 vs 0.2000


In [37]:
# ── 2c: build train/val/test splits and loaders ────────────────────────────────
# Train dataset with random seed=None so each epoch sees different slice positions
# Val/test with fixed seed for reproducible evaluation
full_train_ds = SyntheticDVFDataset(SYNTH_TRAIN_FIELD_DIR, seed=None)
test_ds       = SyntheticDVFDataset(SYNTH_TEST_FIELD_DIR,  seed=42)

val_size   = int(0.15 * len(full_train_ds))   # 75 samples
train_size = len(full_train_ds) - val_size     # 425 samples

train_ds, val_ds = random_split(
    full_train_ds, [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

# batch_size=2 is feasible with 80GB RAM for this volume size
# num_workers=4 for parallel loading
train_loader = DataLoader(train_ds, batch_size=2, shuffle=True,
                          num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=1, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=1, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f'Train: {len(train_ds)} samples, {len(train_loader)} batches')
print(f'Val:   {len(val_ds)} samples,  {len(val_loader)} batches')
print(f'Test:  {len(test_ds)} samples, {len(test_loader)} batches')

# verify one batch loads correctly
dvf_b, cor_b, sag_b, h_b, w_b = next(iter(train_loader))
print(f'\nOne train batch:')
print(f'  dvf: {tuple(dvf_b.shape)}   (expect (2, 3, 256, 256, 128))')
print(f'  cor: {tuple(cor_b.shape)}   (expect (2, 3, 256, 128))')
print(f'  sag: {tuple(sag_b.shape)}   (expect (2, 3, 256, 128))')
print(f'  h:   {h_b}   (expect 2 values in [0,1])')
print(f'  w:   {w_b}   (expect 2 values in [0,1])')

Train: 425 samples, 213 batches
Val:   75 samples,  75 batches
Test:  100 samples, 100 batches

One train batch:
  dvf: (2, 3, 256, 256, 128)   (expect (2, 3, 256, 256, 128))
  cor: (2, 3, 256, 128)   (expect (2, 3, 256, 128))
  sag: (2, 3, 256, 128)   (expect (2, 3, 256, 128))
  h:   tensor([0.7098, 0.4588])   (expect 2 values in [0,1])
  w:   tensor([0.6863, 0.4157])   (expect 2 values in [0,1])


## Section 2 checkpoint
Before continuing, verify:
- dvf shape is (2, 3, 256, 256, 128) in a batch of 2 ✓
- cor and sag shapes are (2, 3, 256, 128) ✓
- h_norm and w_norm vary across samples ✓
- Slice consistency check passed ✓
- dvf range is approx [-1, 1] ✓

## Section 3 — Model architecture

In [43]:
# ── Time + position embedding ──────────────────────────────────────────────────
class SinusoidalEmbedding(nn.Module):
    """Maps a scalar (B,) to a sinusoidal embedding (B, dim)."""
    def __init__(self, dim):
        super().__init__()
        assert dim % 2 == 0
        self.dim = dim

    def forward(self, x):
        # x: (B,) floats
        half  = self.dim // 2
        freqs = torch.exp(
            -math.log(10000) * torch.arange(half, device=x.device) / (half - 1)
        )                          # (half,)
        args  = x[:, None] * freqs[None, :]   # (B, half)
        return torch.cat([torch.sin(args), torch.cos(args)], dim=1)  # (B, dim)


class ConditioningEmbedding(nn.Module):
    """
    Combines:
      - sinusoidal timestep embedding
      - sinusoidal h_norm embedding (coronal slice position)
      - sinusoidal w_norm embedding (sagittal slice position)
    into a single conditioning vector (B, emb_dim).

    This is the mechanism by which the model knows WHERE the conditioning
    slices are. Without this, position is not learnable.
    """
    def __init__(self, emb_dim=128):
        super().__init__()
        self.sin_emb = SinusoidalEmbedding(emb_dim)
        # three emb_dim vectors → one emb_dim vector
        self.mlp = nn.Sequential(
            nn.Linear(emb_dim * 3, emb_dim * 2),
            nn.SiLU(),
            nn.Linear(emb_dim * 2, emb_dim),
        )

    def forward(self, t, h_norm, w_norm):
        """
        t:      (B,) long   — diffusion timestep index
        h_norm: (B,) float  — coronal position in [0,1]
        w_norm: (B,) float  — sagittal position in [0,1]
        returns: (B, emb_dim)
        """
        t_emb = self.sin_emb(t.float())   # (B, emb_dim)
        h_emb = self.sin_emb(h_norm)      # (B, emb_dim)
        w_emb = self.sin_emb(w_norm)      # (B, emb_dim)
        return self.mlp(torch.cat([t_emb, h_emb, w_emb], dim=1))  # (B, emb_dim)

In [44]:
# ── Building blocks ────────────────────────────────────────────────────────────
class ResBlock3D(nn.Module):
    """
    Two conv layers with GroupNorm + SiLU, with a residual connection.
    Time/position conditioning is added via a learned linear projection
    of the embedding vector, broadcast over the spatial dims.
    """
    def __init__(self, in_channels, out_channels, emb_dim):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, in_channels)
        self.conv1 = nn.Conv3d(in_channels, out_channels, 3, padding=1)
        self.norm2 = nn.GroupNorm(8, out_channels)
        self.conv2 = nn.Conv3d(out_channels, out_channels, 3, padding=1)
        self.emb_proj = nn.Linear(emb_dim, out_channels)   # conditioning injection
        self.act      = nn.SiLU()
        # skip connection: 1x1 conv if channel count changes
        if in_channels != out_channels:
            self.skip = nn.Conv3d(in_channels, out_channels, 1)
        else:
            self.skip = nn.Identity()

    def forward(self, x, emb):
        """
        x:   (B, in_channels, D, H, W)
        emb: (B, emb_dim)
        """
        h = self.act(self.norm1(x))
        h = self.conv1(h)
        # add conditioning: project emb to out_channels, broadcast spatially
        h = h + self.emb_proj(self.act(emb))[:, :, None, None, None]
        h = self.act(self.norm2(h))
        h = self.conv2(h)
        return h + self.skip(x)

In [45]:
# ── UNet ───────────────────────────────────────────────────────────────────────
class UNet3D(nn.Module):
    """
    4-level 3D UNet for diffusion on DVF volumes.

    Input convention (Sparse3Diff style):
      x_t: (B, 3, H, W, D) — noisy DVF, but known planes are held at clean values
      The model sees the clean planes directly in x_t and learns to use them.
      No separate conditioning branch is needed.

    Conditioning:
      emb: (B, emb_dim) — from ConditioningEmbedding (timestep + slice positions)
      This is injected at every ResBlock via learned linear projections.

    Channel progression: 3 -> 32 -> 64 -> 128 -> 256 -> bottleneck 256
    Spatial progression: (H,W,D) -> /2 -> /4 -> /8 -> /16 at bottleneck
    With H=256,W=256,D=128: bottleneck spatial = (16, 16, 8)
    All dims divisible by 16: 256/16=16, 128/16=8 — OK.
    """
    def __init__(self, emb_dim=128, base_channels=32):
        super().__init__()
        C  = base_channels
        self.cond_emb = ConditioningEmbedding(emb_dim)

        # ── Encoder ───────────────────────────────────────────────────────────
        self.in_conv = nn.Conv3d(3, C, 3, padding=1)     # 3 -> C

        self.enc1 = ResBlock3D(C,     C,     emb_dim)    # (B, C,   H,   W,   D)
        self.enc2 = ResBlock3D(C,     C*2,   emb_dim)    # after pool: H/2
        self.enc3 = ResBlock3D(C*2,   C*4,   emb_dim)    # after pool: H/4
        self.enc4 = ResBlock3D(C*4,   C*8,   emb_dim)    # after pool: H/8

        self.pool = nn.MaxPool3d(2)

        # ── Bottleneck ────────────────────────────────────────────────────────
        self.mid1 = ResBlock3D(C*8, C*8, emb_dim)       # H/16
        self.mid2 = ResBlock3D(C*8, C*8, emb_dim)

        # ── Decoder (with skip connections) ───────────────────────────────────
        self.up4    = nn.ConvTranspose3d(C*8, C*8, 2, stride=2)
        self.dec4   = ResBlock3D(C*8 + C*8, C*4, emb_dim)   # concat with enc4

        self.up3    = nn.ConvTranspose3d(C*4, C*4, 2, stride=2)
        self.dec3   = ResBlock3D(C*4 + C*4, C*2, emb_dim)   # concat with enc3

        self.up2    = nn.ConvTranspose3d(C*2, C*2, 2, stride=2)
        self.dec2   = ResBlock3D(C*2 + C*2, C,   emb_dim)   # concat with enc2

        self.up1    = nn.ConvTranspose3d(C,   C,   2, stride=2)
        self.dec1   = ResBlock3D(C   + C,   C,   emb_dim)   # concat with enc1

        self.out_conv = nn.Conv3d(C, 3, 1)    # 1x1x1, no activation

    def forward(self, x_t, t, h_norm, w_norm):
        """
        x_t:    (B, 3, H, W, D)  — noisy DVF (known planes are clean)
        t:      (B,) long        — diffusion timestep
        h_norm: (B,) float       — normalised coronal plane position
        w_norm: (B,) float       — normalised sagittal plane position
        returns: (B, 3, H, W, D) — predicted noise
        """
        # permute to (B, C, D, H, W) for 3D convs
        # input is (B, 3, H, W, D), we want depth first for conv efficiency
        x = x_t.permute(0, 1, 4, 2, 3).contiguous()   # (B, 3, D, H, W)

        emb = self.cond_emb(t, h_norm, w_norm)         # (B, emb_dim)

        # encoder
        x  = self.in_conv(x)                           # (B, C, D, H, W)
        e1 = self.enc1(x,             emb)             # (B, C,   D,   H,   W)
        e2 = self.enc2(self.pool(e1), emb)             # (B, C*2, D/2, H/2, W/2)
        e3 = self.enc3(self.pool(e2), emb)             # (B, C*4, D/4, H/4, W/4)
        e4 = self.enc4(self.pool(e3), emb)             # (B, C*8, D/8, H/8, W/8)

        # bottleneck
        m  = self.mid1(self.pool(e4), emb)             # (B, C*8, D/16, H/16, W/16)
        m  = self.mid2(m,             emb)

        # decoder
        d4 = self.dec4(torch.cat([self.up4(m),  e4], dim=1), emb)
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1), emb)
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1), emb)
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1), emb)

        out = self.out_conv(d1)                        # (B, 3, D, H, W)
        return out.permute(0, 1, 3, 4, 2).contiguous()  # (B, 3, H, W, D)

In [91]:
# ── 3a: verify model forward pass and parameter count ─────────────────────────
model = UNet3D(emb_dim=128, base_channels=32).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f'Total parameters: {n_params:,}')

# dummy forward pass with batch size 1
B = 1
x_dummy    = torch.randn(B, 3, H, W, D).to(DEVICE)
t_dummy    = torch.randint(0, 1000, (B,)).to(DEVICE)
h_dummy    = torch.rand(B).to(DEVICE)
w_dummy    = torch.rand(B).to(DEVICE)

with torch.no_grad():
    out = model(x_dummy, t_dummy, h_dummy, w_dummy)

print(f'Input shape:  {tuple(x_dummy.shape)}')
print(f'Output shape: {tuple(out.shape)}   (must match input)')
assert out.shape == x_dummy.shape, f'Shape mismatch: {out.shape} != {x_dummy.shape}'
print('Forward pass: OK')

# memory estimate
if DEVICE == 'cuda':
    print(f'GPU memory after forward: {torch.cuda.memory_allocated()/1e9:.2f} GB')

Total parameters: 14,735,011
Input shape:  (1, 3, 256, 256, 128)
Output shape: (1, 3, 256, 256, 128)   (must match input)
Forward pass: OK
GPU memory after forward: 2.73 GB


## Section 3 checkpoint
Before continuing, verify:
- Output shape matches input (B, 3, 256, 256, 128) ✓
- Forward pass completes without OOM ✓
- Parameter count is printed (expect ~10-20M with base_channels=32) ✓

If you get OOM, reduce base_channels to 24 and rerun.

## Section 4 — Diffusion schedule and loss

In [92]:
class DiffusionSchedule:
    """
    Linear beta schedule.
    All tensors live on the given device.
    """
    def __init__(self, timesteps=1000, beta_start=1e-4, beta_end=0.02, device='cpu'):
        self.timesteps = timesteps
        self.device    = device

        betas     = torch.linspace(beta_start, beta_end, timesteps, device=device)
        alphas    = 1.0 - betas
        alpha_bar = torch.cumprod(alphas, dim=0)

        # precompute values used in forward and reverse process
        self.betas                     = betas
        self.alphas                    = alphas
        self.alpha_bar                 = alpha_bar
        self.sqrt_alpha_bar            = torch.sqrt(alpha_bar)
        self.sqrt_one_minus_alpha_bar  = torch.sqrt(1.0 - alpha_bar)
        # alpha_bar shifted by one step (used in reverse process)
        alpha_bar_prev                 = F.pad(alpha_bar[:-1], (1, 0), value=1.0)
        self.alpha_bar_prev            = alpha_bar_prev

    def q_sample_masked(self, x0, t, noise, mask):
        """
        Sparse3Diff forward process:
          - known planes (mask=1) stay at clean x0 values
          - unknown voxels (mask=0) are noised normally

        x0:    (B, 3, H, W, D)
        t:     (B,) long
        noise: (B, 3, H, W, D)
        mask:  (B, 3, H, W, D) — 1 at known planes, 0 elsewhere
        """
        s_ab  = self.sqrt_alpha_bar[t][:, None, None, None, None]
        s_oab = self.sqrt_one_minus_alpha_bar[t][:, None, None, None, None]
        x_noisy = s_ab * x0 + s_oab * noise
        # known planes always stay clean
        return mask * x0 + (1.0 - mask) * x_noisy

In [93]:
def build_mask(shape, h_idx, w_idx, device):
    """
    Binary mask: 1 at coronal plane h_idx and sagittal plane w_idx, 0 elsewhere.
    h_idx, w_idx are integer indices (not normalised).
    shape: (B, 3, H, W, D)

    Note: h_idx and w_idx can be different per batch item.
    We handle this with a loop — batch size is small so this is fine.
    """
    B, C, H, W, D = shape
    mask = torch.zeros(shape, device=device)
    for b in range(B):
        h = int(h_idx[b])
        w = int(w_idx[b])
        mask[b, :, h, :, :] = 1.0   # coronal plane
        mask[b, :, :, w, :] = 1.0   # sagittal plane
    return mask


def compute_smoothness_loss(pred_x0, mask):
    """
    L2 gradient penalty on the predicted x0, computed only on unknown voxels.
    Encourages spatial smoothness in the reconstructed DVF.
    pred_x0: (B, 3, H, W, D)
    mask:    (B, 3, H, W, D)
    """
    unknown = 1.0 - mask
    f       = pred_x0 * unknown
    dh = (f[:, :, 1:, :,  :] - f[:, :, :-1, :,  :]) ** 2
    dw = (f[:, :, :,  1:, :] - f[:, :, :,  :-1, :]) ** 2
    dd = (f[:, :, :,  :, 1:] - f[:, :, :,  :, :-1]) ** 2
    return (dh.mean() + dw.mean() + dd.mean()) / 3.0


def diffusion_loss(model, schedule, x0, h_norm, w_norm,
                   lambda_smooth=1e-4,
                   lambda_consistency=0.1):
    """
    Three-term loss:
      1. MSE on unknown voxels          — learn to denoise
      2. Consistency on known planes    — force use of conditioning
      3. Smoothness on unknown voxels   — encourage spatial regularity

    lambda_smooth:       weight for smoothness term
    lambda_consistency:  weight for consistency term
    """
    B, C, H_dim, W_dim, D_dim = x0.shape

    h_idx = (h_norm * (H_dim - 1)).round().long()
    w_idx = (w_norm * (W_dim - 1)).round().long()
    mask  = build_mask(x0.shape, h_idx, w_idx, x0.device)

    t     = torch.randint(0, schedule.timesteps, (B,), device=x0.device)
    noise = torch.randn_like(x0)
    x_t   = schedule.q_sample_masked(x0, t, noise, mask)

    noise_pred = model(x_t, t, h_norm, w_norm)

    # ── 1. MSE loss on unknown voxels only ────────────────────────────────────
    unknown  = 1.0 - mask
    n_unk    = unknown.sum().clamp(min=1.0)
    mse_loss = ((noise_pred - noise) ** 2 * unknown).sum() / n_unk

    # ── Predicted x0 (needed for terms 2 and 3) ───────────────────────────────
    s_ab    = schedule.sqrt_alpha_bar[t][:, None, None, None, None]
    s_oab   = schedule.sqrt_one_minus_alpha_bar[t][:, None, None, None, None]
    pred_x0 = (x_t - s_oab * noise_pred) / s_ab.clamp(min=1e-6)
    pred_x0 = pred_x0.clamp(-2.0, 2.0)

    # ── 2. Consistency loss on known planes ───────────────────────────────────
    # Only at timesteps where pred_x0 is reliable (abar > 0.1, i.e. t < ~800)
    # At high noise levels pred_x0 is too noisy to be a useful supervision signal
    abar_scalar        = schedule.alpha_bar[t]              # (B,)
    consistency_weight = (abar_scalar > 0.1).float()        # (B,) — 1 or 0
    consistency_weight = consistency_weight[:, None, None, None, None]
    consistency_loss   = ((pred_x0 - x0) ** 2 * mask * consistency_weight).sum() / \
                         (mask * consistency_weight).sum().clamp(min=1e-8)

    # ── 3. Smoothness loss on unknown voxels only ─────────────────────────────
    # Identical to your original compute_gradient_loss but masked to unknown voxels
    # so we do not penalise gradients at the known plane boundaries
    smooth_loss = compute_smoothness_loss(pred_x0, mask)

    total = mse_loss + lambda_consistency * consistency_loss + \
            lambda_smooth * smooth_loss

    return total, mse_loss.detach(), consistency_loss.detach(), smooth_loss.detach()


def get_smooth_lambda(epoch, total_epochs,
                      initial_lambda=1e-4, final_lambda=1e-6):
    """Cosine decay of smoothness weight after epoch 200."""
    if epoch < 200:
        return initial_lambda
    decay_range = total_epochs - 200
    step        = (initial_lambda - final_lambda) / decay_range
    return max(initial_lambda - step * (epoch - 200), final_lambda)

In [94]:
# ── 4a: verify loss runs and returns sensible values ──────────────────────────
schedule = DiffusionSchedule(timesteps=1000, device=DEVICE)

dvf_b, cor_b, sag_b, h_b, w_b = next(iter(train_loader))
dvf_b = dvf_b.to(DEVICE)
h_b   = h_b.to(DEVICE)
w_b   = w_b.to(DEVICE)

total, mse, consistency, smooth = diffusion_loss(model, schedule, dvf_b, h_b, w_b)
print(f'total loss:        {total.item():.6f}')
print(f'mse loss:          {mse.item():.6f}   (expect ~1.0 at init)')
print(f'consistency loss:  {consistency.item():.6f}   (expect >0)')
print(f'smooth loss:       {smooth.item():.6f}')

# verify gradients flow
total.backward()
grad_norms = [p.grad.norm().item() for p in model.parameters() if p.grad is not None]
print(f'\nGradient norms: min={min(grad_norms):.6f}  max={max(grad_norms):.6f}')
print(f'Parameters with grad: {len(grad_norms)}')
print('Loss and gradient check: OK')

total loss:        1.106777
mse loss:          1.053655   (expect ~1.0 at init)
consistency loss:  0.526535   (expect >0)
smooth loss:       4.691537

Gradient norms: min=0.000071  max=1.384017
Parameters with grad: 130
Loss and gradient check: OK


## Section 4 checkpoint
Before continuing, verify:
- MSE loss is approximately 1.0 at init (random network predicts random noise) ✓
- All parameters have gradients ✓
- No NaN or Inf values ✓

## Section 5 — Sampler

In [95]:
@torch.no_grad()
def sample_sparse3diff(model, schedule, x0_known, h_norm, w_norm, device,
                       n_steps=1000):
    B, C, H_dim, W_dim, D_dim = x0_known.shape

    h_idx = (h_norm * (H_dim - 1)).round().long()
    w_idx = (w_norm * (W_dim - 1)).round().long()
    mask  = build_mask(x0_known.shape, h_idx, w_idx, device)

    # initialise: pure noise for unknown, clean for known
    x = torch.randn_like(x0_known)
    x = mask * x0_known + (1.0 - mask) * x

    timesteps = list(reversed(range(schedule.timesteps)))
    if n_steps < schedule.timesteps:
        indices   = torch.linspace(0, schedule.timesteps - 1,
                                   n_steps).long().tolist()
        timesteps = sorted(indices, reverse=True)

    for t_idx in tqdm(timesteps, desc='Sampling', leave=False):
        tt       = torch.full((B,), t_idx, device=device, dtype=torch.long)
        abar     = schedule.alpha_bar[t_idx]
        abar_prev= schedule.alpha_bar_prev[t_idx]
        beta     = schedule.betas[t_idx]
        alpha    = schedule.alphas[t_idx]

        eps_pred = model(x, tt, h_norm, w_norm)

        # x0 estimate — only reliable at low noise levels
        # at high noise levels, use a weighted combination
        # this is the standard DDPM posterior, written more carefully
        x0_hat = (x - (1.0 - abar).sqrt() * eps_pred) / abar.sqrt().clamp(min=1e-8)

        # clip x0_hat to data range — but use a WIDE clamp
        # because at high t the estimate is noisy but directionally correct
        # the narrow clamp was destroying information
        x0_hat = x0_hat.clamp(-2.0, 2.0)

        # posterior mean: this is the DDPM formula
        # it is numerically stable even at high t because the coefficients
        # naturally downweight x0_hat when abar is small
        coef_x0 = (abar_prev.sqrt() * beta)     / (1.0 - abar)
        coef_xt = (alpha.sqrt() * (1.0 - abar_prev)) / (1.0 - abar)
        mean    = coef_x0 * x0_hat + coef_xt * x

        if t_idx > 0:
            # posterior variance
            var   = beta * (1.0 - abar_prev) / (1.0 - abar)
            x_new = mean + var.sqrt() * torch.randn_like(x)
        else:
            x_new = mean

        # re-anchor known planes at every step
        x = mask * x0_known + (1.0 - mask) * x_new

    return x

In [96]:
# ── 5a: verify sampler runs (fast preview with 50 steps) ──────────────────────
model.eval()
dvf_b, cor_b, sag_b, h_b, w_b = next(iter(val_loader))
dvf_b = dvf_b.to(DEVICE)
h_b   = h_b.to(DEVICE)
w_b   = w_b.to(DEVICE)

with torch.no_grad():
    pred = sample_sparse3diff(model, schedule, dvf_b, h_b, w_b,
                              device=DEVICE, n_steps=50)

print(f'Predicted shape: {tuple(pred.shape)}  (expect (1, 3, 256, 256, 128))')
print(f'Predicted range: [{pred.min():.4f}, {pred.max():.4f}]')
print(f'(At this point it will look like noise — model is not trained yet)')

# verify known planes are exactly preserved
h_idx = int((h_b[0] * (H-1)).round())
w_idx = int((w_b[0] * (W-1)).round())
cor_err = (pred[0, :, h_idx, :, :] - dvf_b[0, :, h_idx, :, :]).abs().max()
sag_err = (pred[0, :, :, w_idx, :] - dvf_b[0, :, :, w_idx, :]).abs().max()
print(f'\nKnown plane preservation:')
print(f'  Coronal  error: {cor_err:.8f}  (must be 0.0)')
print(f'  Sagittal error: {sag_err:.8f}  (must be 0.0)')
model.train()

Sampling:   0%|          | 0/50 [00:00<?, ?it/s]

Predicted shape: (1, 3, 256, 256, 128)  (expect (1, 3, 256, 256, 128))
Predicted range: [-1.9997, 1.9997]
(At this point it will look like noise — model is not trained yet)

Known plane preservation:
  Coronal  error: 0.00000000  (must be 0.0)
  Sagittal error: 0.00000000  (must be 0.0)


UNet3D(
  (cond_emb): ConditioningEmbedding(
    (sin_emb): SinusoidalEmbedding()
    (mlp): Sequential(
      (0): Linear(in_features=384, out_features=256, bias=True)
      (1): SiLU()
      (2): Linear(in_features=256, out_features=128, bias=True)
    )
  )
  (in_conv): Conv3d(3, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
  (enc1): ResBlock3D(
    (norm1): GroupNorm(8, 32, eps=1e-05, affine=True)
    (conv1): Conv3d(32, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
    (norm2): GroupNorm(8, 32, eps=1e-05, affine=True)
    (conv2): Conv3d(32, 32, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
    (emb_proj): Linear(in_features=128, out_features=32, bias=True)
    (act): SiLU()
    (skip): Identity()
  )
  (enc2): ResBlock3D(
    (norm1): GroupNorm(8, 32, eps=1e-05, affine=True)
    (conv1): Conv3d(32, 64, kernel_size=(3, 3, 3), stride=(1, 1, 1), padding=(1, 1, 1))
    (norm2): GroupNorm(8, 64, eps=1e-05, affine=True)
    (conv2): Co

## Section 5 checkpoint
Before continuing, verify:
- Sampler completes with 50 steps ✓
- Known plane errors are exactly 0.0 ✓
- Output shape is correct ✓

## Section 6 — Visualisation helper

In [97]:
def visualise_sample(dvf_gt, dvf_pred, h_norm, w_norm, epoch, save_dir,
                     norm=DVF_NORM, tag=''):
    """
    For one sample (B=1), show GT vs predicted for all 3 components
    in three views: axial (middle D), coronal (at h_idx), sagittal (at w_idx).
    Also marks which planes were used as conditioning.

    dvf_gt, dvf_pred: (1, 3, H, W, D) normalised tensors
    """
    gt   = dvf_gt[0].cpu().numpy()    # (3, H, W, D)
    pred = dvf_pred[0].cpu().numpy()  # (3, H, W, D)

    # denormalise for display
    # norm is (3,) — broadcast over (3, H, W, D)
    gt   = gt   * norm[:, None, None, None]
    pred = pred * norm[:, None, None, None]

    h_idx = int(round(h_norm[0].item() * (H - 1)))
    w_idx = int(round(w_norm[0].item() * (W - 1)))
    d_mid = D // 2

    comp_names = ['dx', 'dy', 'dz']
    view_names = [f'Axial D={d_mid}',
                  f'Coronal H={h_idx} (conditioned)',
                  f'Sagittal W={w_idx} (conditioned)']

    def get_slices(vol):
        # returns list of (H,W), (W,D), (H,D) for each view
        return [
            vol[:, :, :, d_mid],      # axial:    (3, H, W)
            vol[:, h_idx, :, :],      # coronal:  (3, W, D)
            vol[:, :, w_idx, :],      # sagittal: (3, H, D)
        ]

    gt_views   = get_slices(gt)
    pred_views = get_slices(pred)

    n_views = 3
    n_comp  = 3
    fig, axes = plt.subplots(n_views * 2, n_comp,
                             figsize=(n_comp * 4, n_views * 2 * 3))

    for v in range(n_views):
        for c in range(n_comp):
            row_gt   = v * 2
            row_pred = v * 2 + 1

            sl_gt   = gt_views[v][c]
            sl_pred = pred_views[v][c]

            # use GT's 99th percentile for both so colours are comparable
            vmax = max(np.percentile(np.abs(sl_gt), 99), 1.0)

            axes[row_gt, c].imshow(
                sl_gt.T, cmap='RdBu_r', vmin=-vmax, vmax=vmax, origin='lower')
            axes[row_gt, c].set_title(
                f'GT {comp_names[c]} — {view_names[v]}', fontsize=8)
            axes[row_gt, c].axis('off')

            axes[row_pred, c].imshow(
                sl_pred.T, cmap='RdBu_r', vmin=-vmax, vmax=vmax, origin='lower')
            axes[row_pred, c].set_title(
                f'Pred {comp_names[c]} — {view_names[v]}', fontsize=8)
            axes[row_pred, c].axis('off')

    plt.suptitle(f'Epoch {epoch}{tag} — h={h_idx} w={w_idx}', fontsize=11)
    plt.tight_layout()
    fname = os.path.join(save_dir, f'sample_epoch_{epoch:04d}{tag}.png')
    plt.savefig(fname, dpi=100, bbox_inches='tight')
    plt.show()
    plt.close()
    print(f'Saved: {fname}')
    return fname

In [83]:
# ── 6a: verify visualisation helper before training ───────────────────────────
# use GT as both gt and pred to confirm layout is correct
dvf_b, cor_b, sag_b, h_b, w_b = next(iter(val_loader))
dvf_b = dvf_b.to(DEVICE)
h_b   = h_b.to(DEVICE)
w_b   = w_b.to(DEVICE)

# generate a quick sample (50 steps, untrained model = noise)
model.eval()
with torch.no_grad():
    pred = sample_sparse3diff(model, schedule, dvf_b, h_b, w_b,
                              device=DEVICE, n_steps=50)
model.train()

visualise_sample(dvf_b, pred, h_b, w_b, epoch=0, save_dir=VIS_DIR,
                 tag='_pretrain_check')
print('Visualisation helper: OK')
print('Pred should look like noise — model is untrained')

Sampling:   0%|          | 0/50 [00:00<?, ?it/s]

Saved: /mimer/NOBACKUP/groups/caim1/dafne/visual/sample_epoch_0000_pretrain_check.png
Visualisation helper: OK
Pred should look like noise — model is untrained


## Section 6 checkpoint
- Figure saves correctly ✓
- GT panels show recognisable DVF structure ✓
- Pred panels show noise (model not trained yet) ✓
- Conditioned planes are labelled correctly ✓

## Section 7 — Short training run (notebook verification)
Runs for only 5 epochs to verify the full loop works

In [55]:
# ── Re-initialise fresh model and optimizer for clean test ────────────────────
model     = UNet3D(emb_dim=128, base_channels=32).to(DEVICE)
schedule  = DiffusionSchedule(timesteps=1000, device=DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)

LAMBDA_SMOOTH = 1e-4
N_QUICK_EPOCHS = 5

for epoch in range(1, N_QUICK_EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    for dvf_b, cor_b, sag_b, h_b, w_b in tqdm(train_loader,
                                                desc=f'Epoch {epoch}',
                                                leave=False):
        dvf_b = dvf_b.to(DEVICE)
        h_b   = h_b.to(DEVICE)
        w_b   = w_b.to(DEVICE)

        total, mse, smooth = diffusion_loss(
            model, schedule, dvf_b, h_b, w_b, lambda_smooth=LAMBDA_SMOOTH)

        optimizer.zero_grad()
        total.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        epoch_loss += total.item()

    epoch_loss /= len(train_loader)
    print(f'Epoch {epoch:02d} — train loss: {epoch_loss:.5f}')

print('\nShort training run complete. If loss decreased slightly, the loop is working.')

Epoch 1:   0%|          | 0/213 [00:00<?, ?it/s]

Epoch 01 — train loss: 0.25383


Epoch 2:   0%|          | 0/213 [00:00<?, ?it/s]

Epoch 02 — train loss: 0.04911


Epoch 3:   0%|          | 0/213 [00:00<?, ?it/s]

Epoch 03 — train loss: 0.03124


Epoch 4:   0%|          | 0/213 [00:00<?, ?it/s]

Epoch 04 — train loss: 0.02705


Epoch 5:   0%|          | 0/213 [00:00<?, ?it/s]

Epoch 05 — train loss: 0.02106

Short training run complete. If loss decreased slightly, the loop is working.


In [98]:
import gc
gc.collect()
torch.cuda.empty_cache()

# ── Setup ──────────────────────────────────────────────────────────────────────
N_OVERFIT          = 5
N_EPOCHS           = 100
LAMBDA_SMOOTH      = 1e-4
LAMBDA_CONSISTENCY = 0.1

fixed_overfit_ds = SyntheticDVFDataset(SYNTH_TRAIN_FIELD_DIR, seed=123)
overfit_loader   = DataLoader(
    torch.utils.data.Subset(fixed_overfit_ds, list(range(N_OVERFIT))),
    batch_size=1, shuffle=False, num_workers=0
)

model     = UNet3D(emb_dim=128, base_channels=32).to(DEVICE)
schedule  = DiffusionSchedule(timesteps=1000, device=DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.0)

# ── Training ───────────────────────────────────────────────────────────────────
print(f'Overfitting {N_OVERFIT} samples for {N_EPOCHS} epochs')
print(f'lambda_smooth={LAMBDA_SMOOTH}  lambda_consistency={LAMBDA_CONSISTENCY}')
print()
print(f'{"Epoch":>6} | {"total":>8} | {"mse":>8} | {"consist":>8} | {"smooth":>8}')
print('-' * 52)

for epoch in range(1, N_EPOCHS + 1):
    model.train()
    e_total = e_mse = e_cons = e_smooth = 0.0

    for dvf_b, cor_b, sag_b, h_b, w_b in overfit_loader:
        dvf_b = dvf_b.to(DEVICE)
        h_b   = h_b.to(DEVICE)
        w_b   = w_b.to(DEVICE)

        total, mse, consistency, smooth = diffusion_loss(
            model, schedule, dvf_b, h_b, w_b,
            lambda_smooth=LAMBDA_SMOOTH,
            lambda_consistency=LAMBDA_CONSISTENCY)

        optimizer.zero_grad()
        total.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        e_total  += total.item()
        e_mse    += mse.item()
        e_cons   += consistency.item()
        e_smooth += smooth.item()

    n = len(overfit_loader)
    if epoch % 10 == 0:
        print(f'{epoch:>6} | {e_total/n:>8.4f} | {e_mse/n:>8.4f} | '
              f'{e_cons/n:>8.4f} | {e_smooth/n:>8.4f}')

# ── Visualisation ──────────────────────────────────────────────────────────────
print('\nGenerating samples...')
model.eval()

for i, (dvf_b, cor_b, sag_b, h_b, w_b) in enumerate(overfit_loader):
    dvf_b = dvf_b.to(DEVICE)
    h_b   = h_b.to(DEVICE)
    w_b   = w_b.to(DEVICE)

    with torch.no_grad():
        pred = sample_sparse3diff(model, schedule, dvf_b, h_b, w_b,
                                  device=DEVICE, n_steps=200)

    visualise_sample(dvf_b, pred, h_b, w_b,
                     epoch=N_EPOCHS,
                     save_dir=VIS_DIR,
                     tag=f'_overfit_consistency_sample{i}')

print('Done.')

Overfitting 5 samples for 100 epochs
lambda_smooth=0.0001  lambda_consistency=0.1

 Epoch |    total |      mse |  consist |   smooth
----------------------------------------------------
    10 |   0.3379 |   0.3355 |   0.0203 |   3.3701
    20 |   0.1157 |   0.1144 |   0.0087 |   4.1377
    30 |   0.1795 |   0.1792 |   0.0000 |   2.8841
    40 |   0.0720 |   0.0665 |   0.0524 |   2.5219
    50 |   0.0534 |   0.0486 |   0.0483 |   0.3362
    60 |   0.0497 |   0.0491 |   0.0028 |   2.6061
    70 |   0.0342 |   0.0332 |   0.0078 |   2.0324
    80 |   0.0396 |   0.0376 |   0.0191 |   0.7775
    90 |   0.0579 |   0.0549 |   0.0288 |   1.4728
   100 |   0.0260 |   0.0241 |   0.0162 |   2.7396

Generating samples...


Sampling:   0%|          | 0/200 [00:00<?, ?it/s]

Saved: /mimer/NOBACKUP/groups/caim1/dafne/visual/sample_epoch_0100_overfit_consistency_sample0.png


Sampling:   0%|          | 0/200 [00:00<?, ?it/s]

Saved: /mimer/NOBACKUP/groups/caim1/dafne/visual/sample_epoch_0100_overfit_consistency_sample1.png


...

Saved: /mimer/NOBACKUP/groups/caim1/dafne/visual/sample_epoch_0100_overfit_consistency_sample2.png


Sampling:   0%|          | 0/200 [00:00<?, ?it/s]

Saved: /mimer/NOBACKUP/groups/caim1/dafne/visual/sample_epoch_0100_overfit_consistency_sample3.png


Sampling:   0%|          | 0/200 [00:00<?, ?it/s]

Saved: /mimer/NOBACKUP/groups/caim1/dafne/visual/sample_epoch_0100_overfit_consistency_sample4.png
Done.


In [88]:
# Step 3: visualise
visualise_sample(dvf_b, result, h_b, w_b,
                 epoch=100, save_dir=VIS_DIR,
                 tag='_fixed_sampler')

Saved: /mimer/NOBACKUP/groups/caim1/dafne/visual/sample_epoch_0100_fixed_sampler.png


'/mimer/NOBACKUP/groups/caim1/dafne/visual/sample_epoch_0100_fixed_sampler.png'

In [99]:
import gc
gc.collect()
torch.cuda.empty_cache()

# ── Setup ──────────────────────────────────────────────────────────────────────
N_OVERFIT     = 5
N_EPOCHS      = 100
LAMBDA_SMOOTH = 1e-4

fixed_overfit_ds = SyntheticDVFDataset(SYNTH_TRAIN_FIELD_DIR, seed=123)
overfit_loader   = DataLoader(
    torch.utils.data.Subset(fixed_overfit_ds, list(range(N_OVERFIT))),
    batch_size=1, shuffle=False, num_workers=0
)

model     = UNet3D(emb_dim=128, base_channels=32).to(DEVICE)
schedule  = DiffusionSchedule(timesteps=1000, device=DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.0)

# ── Train ───────────────────────────────────────────────────────────────────────
print(f'{"Epoch":>6} | {"total":>8} | {"mse":>8} | {"smooth":>8}')
print('-' * 40)

for epoch in range(1, N_EPOCHS + 1):
    model.train()
    e_total = e_mse = e_smooth = 0.0
    for dvf_b, cor_b, sag_b, h_b, w_b in overfit_loader:
        dvf_b = dvf_b.to(DEVICE)
        h_b   = h_b.to(DEVICE)
        w_b   = w_b.to(DEVICE)
        total, mse, consistency, smooth = diffusion_loss(
            model, schedule, dvf_b, h_b, w_b,
            lambda_smooth=LAMBDA_SMOOTH,
            lambda_consistency=0.0)
        optimizer.zero_grad()
        total.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        e_total  += total.item()
        e_mse    += mse.item()
        e_smooth += smooth.item()
    n = len(overfit_loader)
    if epoch % 10 == 0:
        print(f'{epoch:>6} | {e_total/n:>8.4f} | {e_mse/n:>8.4f} | '
              f'{e_smooth/n:>8.4f}')

# ── Sample with 200 steps then 1000 steps — compare ───────────────────────────
print('\nSampling...')
model.eval()
dvf_b, cor_b, sag_b, h_b, w_b = next(iter(overfit_loader))
dvf_b = dvf_b.to(DEVICE)
h_b   = h_b.to(DEVICE)
w_b   = w_b.to(DEVICE)

with torch.no_grad():
    pred_200 = sample_sparse3diff(model, schedule, dvf_b, h_b, w_b,
                                  device=DEVICE, n_steps=200)
    pred_1000 = sample_sparse3diff(model, schedule, dvf_b, h_b, w_b,
                                   device=DEVICE, n_steps=1000)

visualise_sample(dvf_b, pred_200, h_b, w_b,
                 epoch=100, save_dir=VIS_DIR,
                 tag='_200steps')
visualise_sample(dvf_b, pred_1000, h_b, w_b,
                 epoch=100, save_dir=VIS_DIR,
                 tag='_1000steps')
print('Done. Upload both images.')

 Epoch |    total |      mse |   smooth
----------------------------------------
    10 |   0.2853 |   0.2849 |   3.6157
    20 |   0.1069 |   0.1066 |   2.9743
    30 |   0.0940 |   0.0938 |   2.6135
    40 |   0.0947 |   0.0944 |   2.6604
    50 |   0.1983 |   0.1982 |   0.4728
    60 |   0.0841 |   0.0840 |   0.8046
    70 |   0.0385 |   0.0385 |   0.4454
    80 |   0.1572 |   0.1571 |   1.8363
    90 |   0.0310 |   0.0309 |   0.4093
   100 |   0.1783 |   0.1781 |   1.9875

Sampling...


Sampling:   0%|          | 0/200 [00:00<?, ?it/s]

Sampling:   0%|          | 0/1000 [00:00<?, ?it/s]

Saved: /mimer/NOBACKUP/groups/caim1/dafne/visual/sample_epoch_0100_200steps.png
Saved: /mimer/NOBACKUP/groups/caim1/dafne/visual/sample_epoch_0100_1000steps.png
Done. Upload both images.


In [68]:
# How much does the axial prediction vary across multiple samples 
# from the same trained model?
# If it varies a lot → the model is uncertain (hypothesis 1 or 3)
# If it barely varies → the model is confidently predicting the wrong thing (hypothesis 2)

model.eval()
dvf_b, cor_b, sag_b, h_b, w_b = next(iter(overfit_loader))
dvf_b = dvf_b.to(DEVICE)
h_b   = h_b.to(DEVICE)
w_b   = w_b.to(DEVICE)

preds = []
with torch.no_grad():
    for i in range(5):
        pred = sample_sparse3diff(model, schedule, dvf_b, h_b, w_b,
                                  device=DEVICE, n_steps=200)
        preds.append(pred.cpu())

preds  = torch.stack(preds)   # (5, 1, 3, H, W, D)
d_mid  = D // 2

# std across 5 samples at the axial slice
pred_std = preds[:, 0, :, :, :, d_mid].std(dim=0)   # (3, H, W)

# std of GT at the same slice (should be 0 — it is fixed)
gt_axial = dvf_b[0, :, :, :, d_mid].cpu()            # (3, H, W)

print('Variance across 5 independent samples (axial slice):')
for c in range(3):
    print(f'  channel {c}: pred std = {pred_std[c].mean():.4f}  '
          f'gt value std = {gt_axial[c].std():.4f}')

print('\nIf pred std >> 0: model is uncertain about axial (hypothesis 1 or 3)')
print('If pred std ~= 0: model is confidently wrong (hypothesis 2)')

Sampling:   0%|          | 0/200 [00:00<?, ?it/s]

Sampling:   0%|          | 0/200 [00:00<?, ?it/s]

Sampling:   0%|          | 0/200 [00:00<?, ?it/s]

Sampling:   0%|          | 0/200 [00:00<?, ?it/s]

Sampling:   0%|          | 0/200 [00:00<?, ?it/s]

Variance across 5 independent samples (axial slice):
  channel 0: pred std = 0.6708  gt value std = 0.1223
  channel 1: pred std = 0.6700  gt value std = 0.1599
  channel 2: pred std = 0.6777  gt value std = 0.0972

If pred std >> 0: model is uncertain about axial (hypothesis 1 or 3)
If pred std ~= 0: model is confidently wrong (hypothesis 2)


In [69]:
# Test: what happens if we give the model the WRONG position embedding
# but the same x_t?
# If the output changes a lot → position embedding is being used
# If the output barely changes → position embedding is being ignored

model.eval()
dvf_b, cor_b, sag_b, h_b, w_b = next(iter(overfit_loader))
dvf_b = dvf_b.to(DEVICE)
h_b   = h_b.to(DEVICE)
w_b   = w_b.to(DEVICE)

# build x_t at a fixed mid timestep
t_idx  = 500
tt     = torch.full((1,), t_idx, device=DEVICE, dtype=torch.long)
noise  = torch.randn_like(dvf_b)
h_idx  = (h_b * (H - 1)).round().long()
w_idx  = (w_b * (W - 1)).round().long()
mask   = build_mask(dvf_b.shape, h_idx, w_idx, DEVICE)
x_t    = schedule.q_sample_masked(dvf_b, tt, noise, mask)

with torch.no_grad():
    # correct position
    eps_correct = model(x_t, tt, h_b, w_b)
    
    # wrong position — tell the model the planes are at 0.5, 0.5
    # regardless of where they actually are
    h_wrong = torch.full_like(h_b, 0.5)
    w_wrong = torch.full_like(w_b, 0.5)
    eps_wrong = model(x_t, tt, h_wrong, w_wrong)
    
    # random position
    h_rand = torch.rand_like(h_b)
    w_rand = torch.rand_like(w_b)
    eps_rand = model(x_t, tt, h_rand, w_rand)

diff_wrong = (eps_correct - eps_wrong).abs()
diff_rand  = (eps_correct - eps_rand).abs()

print('Effect of changing position embedding on noise prediction:')
print(f'  Mean abs diff (correct vs wrong pos): {diff_wrong.mean():.6f}')
print(f'  Mean abs diff (correct vs rand  pos): {diff_rand.mean():.6f}')
print(f'  Noise pred std (correct pos):         {eps_correct.std():.6f}')
print()
print('If diff << noise pred std: position embedding is being IGNORED')
print('If diff ~= noise pred std: position embedding is being USED')

Effect of changing position embedding on noise prediction:
  Mean abs diff (correct vs wrong pos): 0.001149
  Mean abs diff (correct vs rand  pos): 0.003086
  Noise pred std (correct pos):         0.992231

If diff << noise pred std: position embedding is being IGNORED
If diff ~= noise pred std: position embedding is being USED


In [71]:
model_test = UNet3D(emb_dim=128, base_channels=32).to(DEVICE)
optimizer_test = torch.optim.AdamW(model_test.parameters(), lr=1e-4)

dvf_b, cor_b, sag_b, h_b, w_b = next(iter(overfit_loader))
dvf_b = dvf_b.to(DEVICE)
h_b   = h_b.to(DEVICE)
w_b   = w_b.to(DEVICE)

# check consistency loss value at init
total, mse, consistency, smooth = diffusion_loss(
    model_test, schedule, dvf_b, h_b, w_b,
    lambda_smooth=0.0, lambda_consistency=1.0)

print(f'At init:')
print(f'  mse_loss:         {mse.item():.6f}  (expect ~1.0)')
print(f'  consistency_loss: {consistency.item():.6f}  (expect >0)')
print(f'  total:            {total.item():.6f}')

# verify gradient flows through consistency loss
total.backward()
grad_norms = [p.grad.norm().item() 
              for p in model_test.parameters() if p.grad is not None]
print(f'  max gradient norm: {max(grad_norms):.6f}  (must be >0)')

# now check: does the model learn to use known planes faster with consistency?
print('\nTraining 20 steps with consistency loss:')
optimizer_test.zero_grad()
for step in range(20):
    total, mse, cons, smooth = diffusion_loss(
        model_test, schedule, dvf_b, h_b, w_b,
        lambda_smooth=0.0, lambda_consistency=1.0)
    optimizer_test.zero_grad()
    total.backward()
    optimizer_test.step()
    if step % 5 == 0:
        print(f'  step {step:02d}: mse={mse.item():.4f}  consistency={cons.item():.4f}')

# check if model now uses known planes
with torch.no_grad():
    t_idx = 500
    tt    = torch.full((1,), t_idx, device=DEVICE, dtype=torch.long)
    noise = torch.randn_like(dvf_b)
    h_idx = (h_b * (H-1)).round().long()
    w_idx = (w_b * (W-1)).round().long()
    mask  = build_mask(dvf_b.shape, h_idx, w_idx, DEVICE)
    x_t   = schedule.q_sample_masked(dvf_b, tt, noise, mask)

    eps_normal = model_test(x_t, tt, h_b, w_b)
    x_t_noised = (1.0 - mask) * x_t + mask * torch.randn_like(x_t)
    eps_noised = model_test(x_t_noised, tt, h_b, w_b)

diff = (eps_normal - eps_noised).abs()
print(f'\nAfter 20 steps with consistency loss:')
print(f'  Mean abs diff (normal vs noised planes): {diff.mean():.6f}')
print(f'  Noise pred std:                          {eps_normal.std():.6f}')
print(f'  Ratio: {diff.mean()/eps_normal.std():.4f}  (was 0.0109 before, higher is better)')

TypeError: diffusion_loss() got an unexpected keyword argument 'lambda_consistency'

## Section 7 checkpoint
- 5 epochs complete without OOM ✓
- Loss decreases (even slightly) over 5 epochs ✓
- No NaN values in loss ✓

If all checkpoints pass, we are ready to write the training script.